# Retrieval RAGAS Eval

Run retrieval-only evaluation on `data/evals/retrieval/text/aapl_2024_10k_text_retrieval_eval_split_with_uids.jsonl` using:

- Deterministic metrics (`hit@k`, `recall@k`, `mrr@k`, `ndcg@k`)
- Optional RAGAS context metrics with local Ollama judge


In [1]:
# import sys
# print(sys.executable)

# # Then check/install in that same kernel:

# %pip show ragas
# %pip install ragas langchain langchain-ollama datasets

import ragas
print(ragas.__version__)

0.4.3


In [2]:
from __future__ import annotations

import os
import sys
import json
import tempfile
from pathlib import Path

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not resolve repo root with src/ and notebooks/')

REPO_ROOT = resolve_repo_root()
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('REPO_ROOT =', REPO_ROOT)
print('SRC_PATH  =', SRC_PATH)


REPO_ROOT = /Users/shicheny/Documents/GitHub/FinSearch-reboot
SRC_PATH  = /Users/shicheny/Documents/GitHub/FinSearch-reboot/src


In [3]:
from evals.ragas_retrieval_metrics import RagasRetrievalConfig
from evals.retrieval_eval_runner import run_retrieval_eval


In [4]:
# --- Config ---
EVAL_PATH = REPO_ROOT / 'data' / 'evals' / 'retrieval' / 'text' / 'aapl_2024_10k_text_retrieval_eval_split_with_uids.jsonl'
OUT_DIR = REPO_ROOT / 'artifacts' / 'evals' / 'retrieval' / 'v0'
EVAL_MODE = 'text'  # auto|table|text

DEFAULT_TICKER = 'AAPL'
DEFAULT_FISCAL_YEAR = 2024
DEFAULT_FORM_TYPE = '10-K'
DOC_TYPES = ['text_chunk']

TOP_K = 10
K_VALUES = [1, 3, 5, 10]
MIN_TOTAL_SCORE = 0

# Set SUBSET_N = None to run all rows in the selected eval JSONL
SUBSET_N = None

# Toggle RAGAS in notebook runs
ENABLE_RAGAS = True
# Context recall is parser-sensitive on local Ollama judges; keep off by default.
RAGAS_ENABLE_CONTEXT_RECALL = False

# Subprocess mode avoids notebook event-loop conflicts from ragas async internals.
RUN_EVAL_IN_SUBPROCESS = True

# Use installed local Ollama models by default.
RAGAS_JUDGE_MODEL = os.getenv('RAGAS_JUDGE_MODEL', 'qwen3:4b-instruct')
RAGAS_EMBED_MODEL = os.getenv('RAGAS_EMBED_MODEL', 'nomic-embed-text')

# Print raw subprocess stdout/stderr (can be noisy).
EVAL_DEBUG_OUTPUT = False


In [5]:
def make_subset_eval_file(src_path: Path, n: int) -> Path:
    safe_stem = src_path.stem.replace(' ', '_')
    out_path = Path(tempfile.gettempdir()) / f'{safe_stem}_subset_{n}.jsonl'
    with src_path.open('r', encoding='utf-8') as src, out_path.open('w', encoding='utf-8') as dst:
        for i, line in enumerate(src):
            if i >= n:
                break
            dst.write(line)
    return out_path

eval_path_for_run = EVAL_PATH
if SUBSET_N is not None:
    eval_path_for_run = make_subset_eval_file(EVAL_PATH, SUBSET_N)

print('eval_path_for_run =', eval_path_for_run)


eval_path_for_run = /Users/shicheny/Documents/GitHub/FinSearch-reboot/data/evals/retrieval/text/aapl_2024_10k_text_retrieval_eval_split_with_uids.jsonl


In [6]:
# Optional quick preflight checks
import requests

qdrant_host = os.getenv('QDRANT_HOST', 'localhost')
qdrant_port = os.getenv('QDRANT_PORT', '6333')
ollama_base = os.getenv('RAGAS_OLLAMA_BASE_URL', 'http://localhost:11434')

def check_http(url: str, timeout: int = 3):
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code, None
    except Exception as e:
        return None, str(e)

qdrant_status, qdrant_err = check_http(f'http://{qdrant_host}:{qdrant_port}/collections')
ollama_status, ollama_err = check_http(f'{ollama_base}/api/tags')

print('Qdrant  /collections:', qdrant_status, qdrant_err)
print('Ollama  /api/tags   :', ollama_status, ollama_err)

available_models = []
if ollama_status == 200:
    try:
        tags = requests.get(f"{ollama_base}/api/tags", timeout=5).json()
        available_models = [m.get('name') for m in tags.get('models', []) if isinstance(m, dict) and m.get('name')]
    except Exception as e:
        print('Failed to parse /api/tags:', e)

print('Available Ollama models:', available_models)


Qdrant  /collections: 200 None
Ollama  /api/tags   : 200 None
Available Ollama models: ['qwen3:8b', 'gemma3:4b', 'qwen2.5-coder:7b', 'gemini-3-pro-preview:latest', 'gemini-3-flash-preview:cloud', 'qwen3:4b-instruct', 'qwen2.5:14b-instruct', 'qwen2.5:7b-instruct', 'qwen2.5:14b', 'qwen2.5:7b', 'qwen3-embedding:8b', 'kimi-k2-thinking:cloud', 'kimi-k2:1t-cloud', 'minimax-m2:cloud', 'deepseek-v3.1:671b-cloud', 'gpt-oss:120b-cloud', 'gpt-oss:20b-cloud', 'glm-4.6:cloud', 'deepseek-r1:14b', 'qwen3:14b', 'llama3.1:8b', 'qwen3:latest', 'nomic-embed-text:latest']


In [7]:
import subprocess

cmd = [
    sys.executable,
    'scripts/evals/eval_retrieval_v0.py',
    '--eval-path', str(eval_path_for_run),
    '--out-dir', str(OUT_DIR),
    '--eval-mode', EVAL_MODE,
    '--top-k', str(TOP_K),
    '--k-values', ','.join(str(k) for k in K_VALUES),
    '--doc-types', ','.join(DOC_TYPES),
    '--min-total-score', str(MIN_TOTAL_SCORE),
    '--default-ticker', DEFAULT_TICKER,
    '--default-fiscal-year', str(DEFAULT_FISCAL_YEAR),
    '--default-form-type', DEFAULT_FORM_TYPE,
    '--ragas-judge-model', RAGAS_JUDGE_MODEL,
    '--ragas-embed-model', RAGAS_EMBED_MODEL,
]

if not ENABLE_RAGAS:
    cmd.append('--disable-ragas')

if RAGAS_ENABLE_CONTEXT_RECALL:
    cmd.append('--enable-context-recall')

# The runner currently toggles recall from backend defaults; keep notebook guard here.
if RAGAS_ENABLE_CONTEXT_RECALL:
    print('Note: context_recall enabled; parser failures may still occur with local models.')

env = os.environ.copy()
env['PYTHONPATH'] = str(SRC_PATH)

def _run_eval_subprocess():
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=env,
        text=True,
        capture_output=True,
    )
    if EVAL_DEBUG_OUTPUT and proc.stdout:
        print(proc.stdout)
    if EVAL_DEBUG_OUTPUT and proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f'eval_retrieval_v0.py failed with exit code {proc.returncode}')

if RUN_EVAL_IN_SUBPROCESS:
    _run_eval_subprocess()
else:
    raise RuntimeError('RUN_EVAL_IN_SUBPROCESS=False is not supported in this notebook mode.')

summary_path = OUT_DIR / 'summary.json'
per_query_path = OUT_DIR / 'per_query.jsonl'
errors_path = OUT_DIR / 'errors.jsonl'

summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
rows = []
if per_query_path.exists():
    with per_query_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

errors = []
if errors_path.exists():
    with errors_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                errors.append(json.loads(line))

print(f'Loaded summary/rows/errors from artifacts: rows={len(rows)} errors={len(errors)}')


Loaded summary/rows/errors from artifacts: rows=75 errors=0


In [16]:
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "num_queries": 75,
  "num_valid_queries": 75,
  "num_failures": 0,
  "deterministic": {
    "hit@1": 0.9733333333333334,
    "hit@10": 1.0,
    "hit@3": 1.0,
    "hit@5": 1.0,
    "mrr@1": 0.9733333333333334,
    "mrr@10": 0.9866666666666667,
    "mrr@3": 0.9866666666666667,
    "mrr@5": 0.9866666666666667,
    "ndcg@1": 0.9733333333333334,
    "ndcg@10": 0.9843493114573312,
    "ndcg@3": 0.986946958327833,
    "ndcg@5": 0.9848760689788626,
    "recall@1": 0.8833333333333333,
    "recall@10": 0.9955555555555556,
    "recall@3": 0.9822222222222223,
    "recall@5": 0.9922222222222223
  },
  "ragas": {
    "context_precision": NaN
  },
  "config": {
    "eval_path": "/Users/shicheny/Documents/GitHub/FinSearch-reboot/data/evals/retrieval/text/aapl_2024_10k_text_retrieval_eval_split_with_uids.jsonl",
    "top_k": 10,
    "k_values": [
      1,
      3,
      5,
      10
    ],
    "eval_mode": "text",
    "default_ticker": "AAPL",
    "default_fiscal_year": 2024,
    "default_form_type"

In [9]:
import pandas as pd

records = []
for r in rows:
    rd = r if isinstance(r, dict) else r.model_dump(mode='json')
    ragas = rd.get('ragas') or {}
    metrics = rd.get('metrics') or {}
    retrieve_ms = ((rd.get('trace') or {}).get('timing_ms') or {}).get('retrieve')
    records.append({
        'id': rd.get('id'),
        'query': rd.get('query'),
        'retrieval_ok': rd.get('retrieval_ok'),
        'retrieval_error': rd.get('retrieval_error'),
        'hit@1': metrics.get('hit@1'),
        'hit@3': metrics.get('hit@3'),
        'recall@3': metrics.get('recall@3'),
        'mrr@3': metrics.get('mrr@3'),
        'ndcg@3': metrics.get('ndcg@3'),
        'context_precision': ragas.get('context_precision'),
        'context_recall': ragas.get('context_recall'),
        'retrieve_ms': retrieve_ms,
    })

df = pd.DataFrame(records)
df


,id,query,retrieval_ok,retrieval_error,hit@1,hit@3,recall@3,mrr@3,ndcg@3,context_precision,context_recall,retrieve_ms
0,AAPL10K24_TXT_001,"What does Apple say it designs, manufactures a...",True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,4405
1,AAPL10K24_TXT_002,Apple uses a 52- or 53-week fiscal year. When ...,True,None,0.0,1.0,1.000000,0.5,0.630930,NaN,None,2944
2,AAPL10K24_TXT_003,List the major product categories Apple report...,True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,2879
3,AAPL10K24_TXT_004,"In Apple’s product definitions, what does 'Wea...",True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,2944
4,AAPL10K24_TXT_005,What kinds of Services does Apple describe (pl...,True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,2870
...,...,...,...,...,...,...,...,...,...,...,...,...
70,AAPL10K24_TXT_071,Capital return funding: Where does Apple discu...,True,None,1.0,1.0,0.666667,1.0,0.919721,NaN,None,3626
71,AAPL10K24_TXT_072,Geographic segmentation clarity: Which section...,True,None,1.0,1.0,1.000000,1.0,0.919721,NaN,None,3518
72,AAPL10K24_TXT_073,Product cycle risk: What does Apple say about ...,True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,3680
73,AAPL10K24_TXT_074,Uncertain tax positions deep-dive: What does A...,True,None,1.0,1.0,1.000000,1.0,1.000000,NaN,None,3681


In [10]:
print('Artifacts written to:', OUT_DIR)
print('summary.json exists:', (OUT_DIR / 'summary.json').exists())
print('per_query.jsonl exists:', (OUT_DIR / 'per_query.jsonl').exists())
print('errors.jsonl exists:', (OUT_DIR / 'errors.jsonl').exists())

errors[:5]


Artifacts written to: /Users/shicheny/Documents/GitHub/FinSearch-reboot/artifacts/evals/retrieval/v0
summary.json exists: True
per_query.jsonl exists: True
errors.jsonl exists: True


[]

In [11]:
# Overall / average metrics across retrieved records
metric_cols = [
    'hit@1',
    'hit@3',
    'recall@3',
    'mrr@3',
    'ndcg@3',
    'context_precision',
    'context_recall',
    'retrieve_ms',
]

metrics_df = df.copy()
for c in metric_cols:
    if c in metrics_df.columns:
        metrics_df[c] = pd.to_numeric(metrics_df[c], errors='coerce')

overall_stats = {
    'num_records': int(len(metrics_df)),
    'num_retrieval_ok': int(metrics_df['retrieval_ok'].fillna(False).sum()) if 'retrieval_ok' in metrics_df.columns else 0,
    'retrieval_success_rate': float(metrics_df['retrieval_ok'].fillna(False).mean()) if 'retrieval_ok' in metrics_df.columns and len(metrics_df) > 0 else 0.0,
}

avg_metrics = metrics_df[[c for c in metric_cols if c in metrics_df.columns]].mean(skipna=True)

print('overall_stats:')
print(json.dumps(overall_stats, indent=2))
print('\navg_metrics:')
display(avg_metrics.to_frame('avg').T)


overall_stats:
{
  "num_records": 75,
  "num_retrieval_ok": 75,
  "retrieval_success_rate": 1.0
}

avg_metrics:


,hit@1,hit@3,recall@3,mrr@3,ndcg@3,context_precision,context_recall,retrieve_ms
avg,0.973333,1.0,0.982222,0.986667,0.986947,1.0,NaN,3371.92


In [17]:
# Drill-down helper: inspect top retrieved results for a weak query (low mrr/ndcg)
import re
from qdrant_client import QdrantClient, models

# 1) Quick shortlist of weaker rows
metric_view_cols = [c for c in ["id", "query", "mrr@3", "ndcg@3", "hit@1", "hit@3", "recall@3"] if c in df.columns]
ranked_df = df[metric_view_cols].sort_values(by=["mrr@3", "ndcg@3"], ascending=[True, True]).reset_index(drop=True)
display(ranked_df.head(15))

# 2) Choose the query you want to inspect
TARGET_QUERY_ID = "AAPL10K24_TXT_002"         # e.g. "AAPL10K24_TXT_010"
TARGET_QUERY_SUBSTRING = None  # e.g. "services"
TARGET_ROW_INDEX = None        # index from ranked_df above; e.g. 0 for worst row
TOP_N = 3                      # set to 3 or 5
CONTENT_CHARS = 500
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME", "sec_docs_hybrid")

row_dicts = [r if isinstance(r, dict) else r.model_dump(mode="json") for r in rows]

def _pick_target_row():
    if TARGET_QUERY_ID is not None:
        for rd in row_dicts:
            if str(rd.get("id")) == str(TARGET_QUERY_ID):
                return rd
        raise ValueError(f"TARGET_QUERY_ID not found: {TARGET_QUERY_ID}")

    if TARGET_QUERY_SUBSTRING:
        q = str(TARGET_QUERY_SUBSTRING).lower()
        for rd in row_dicts:
            if q in str(rd.get("query", "")).lower():
                return rd
        raise ValueError(f"No query matched substring: {TARGET_QUERY_SUBSTRING}")

    if TARGET_ROW_INDEX is not None:
        rid = ranked_df.loc[int(TARGET_ROW_INDEX), "id"]
        for rd in row_dicts:
            if str(rd.get("id")) == str(rid):
                return rd
        raise ValueError(f"Could not map TARGET_ROW_INDEX={TARGET_ROW_INDEX} to row id")

    # default: pick the lowest mrr/ndcg row
    rid = ranked_df.loc[0, "id"]
    for rd in row_dicts:
        if str(rd.get("id")) == str(rid):
            return rd
    raise RuntimeError("Failed to pick a target row")

def _parse_table_index(doc_id: str):
    m = re.search(r"::table::(\\d+)", str(doc_id))
    return int(m.group(1)) if m else None

def _gold_match(doc_id: str, relevant_text_doc_ids, relevant_table_indices):
    if relevant_text_doc_ids:
        return str(doc_id) in {str(x) for x in relevant_text_doc_ids}
    if relevant_table_indices:
        idx = _parse_table_index(doc_id)
        if idx is None:
            return False
        return idx in {int(x) for x in relevant_table_indices}
    return None

target = _pick_target_row()
print("target_id:", target.get("id"))
print("target_query:", target.get("query"))
print("target_metrics:", json.dumps(target.get("metrics", {}), indent=2))

retrieved_doc_ids = list(target.get("retrieved_doc_ids") or [])
if not retrieved_doc_ids:
    print("No retrieved_doc_ids found for this row.")
else:
    top_doc_ids = retrieved_doc_ids[:int(TOP_N)]
    relevant_text_doc_ids = list(target.get("relevant_text_doc_ids") or [])
    relevant_table_indices = list(target.get("relevant_table_indices") or [])

    client = QdrantClient(
        host=os.getenv("QDRANT_HOST", "localhost"),
        port=int(os.getenv("QDRANT_PORT", "6333")),
    )

    def _fetch_payload(doc_id: str):
        flt = models.Filter(
            must=[models.FieldCondition(key="doc_id", match=models.MatchValue(value=doc_id))]
        )
        points, _ = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=flt,
            limit=1,
            with_payload=True,
            with_vectors=False,
        )
        if points:
            return points[0].payload or {}
        return {}

    details = []
    for rank, doc_id in enumerate(top_doc_ids, start=1):
        payload = _fetch_payload(doc_id)
        content = (
            payload.get("content")
            or payload.get("rerank_original_content")
            or payload.get("rerank_table_summary")
            or ""
        )
        snippet = str(content).replace("\\n", " ")[:int(CONTENT_CHARS)]

        details.append(
            {
                "rank": rank,
                "doc_id": doc_id,
                "gold_match": _gold_match(doc_id, relevant_text_doc_ids, relevant_table_indices),
                "doc_type": payload.get("doc_type"),
                "ticker": payload.get("ticker"),
                "fiscal_year": payload.get("fiscal_year"),
                "form_type": payload.get("form_type"),
                "table_index": payload.get("table_index"),
                "section_title": payload.get("section_title"),
                "section_path": payload.get("section_path"),
                "chunk_uid": payload.get("chunk_uid"),
                "snippet": snippet,
            }
        )

    details_df = pd.DataFrame(details)
    display(details_df)


,id,query,mrr@3,ndcg@3,hit@1,hit@3,recall@3
0,AAPL10K24_TXT_002,Apple uses a 52- or 53-week fiscal year. When ...,0.5,0.630930,0.0,1.0,1.000000
1,AAPL10K24_TXT_011,What does Apple disclose about supply risk fro...,0.5,0.630930,0.0,1.0,1.000000
2,AAPL10K24_TXT_069,Supply chain concentration: What does Apple sa...,1.0,0.919721,1.0,1.0,1.000000
3,AAPL10K24_TXT_071,Capital return funding: Where does Apple discu...,1.0,0.919721,1.0,1.0,0.666667
4,AAPL10K24_TXT_072,Geographic segmentation clarity: Which section...,1.0,0.919721,1.0,1.0,1.000000
5,AAPL10K24_TXT_001,"What does Apple say it designs, manufactures a...",1.0,1.000000,1.0,1.0,1.000000
6,AAPL10K24_TXT_003,List the major product categories Apple report...,1.0,1.000000,1.0,1.0,1.000000
7,AAPL10K24_TXT_004,"In Apple’s product definitions, what does 'Wea...",1.0,1.000000,1.0,1.0,1.000000
8,AAPL10K24_TXT_005,What kinds of Services does Apple describe (pl...,1.0,1.000000,1.0,1.0,1.000000
9,AAPL10K24_TXT_006,Which payment-related services are mentioned i...,1.0,1.000000,1.0,1.0,1.000000


target_id: AAPL10K24_TXT_002
target_query: Apple uses a 52- or 53-week fiscal year. When is an extra week added, and which of FY2022–FY2024 were 52 vs 53 weeks?
target_metrics: {
  "hit@1": 0.0,
  "recall@1": 0.0,
  "mrr@1": 0.0,
  "ndcg@1": 0.0,
  "hit@3": 1.0,
  "recall@3": 1.0,
  "mrr@3": 0.5,
  "ndcg@3": 0.6309297535714575,
  "hit@5": 1.0,
  "recall@5": 1.0,
  "mrr@5": 0.5,
  "ndcg@5": 0.6309297535714575,
  "hit@10": 1.0,
  "recall@10": 1.0,
  "mrr@10": 0.5,
  "ndcg@10": 0.6309297535714575
}


,rank,doc_id,gold_match,doc_type,ticker,fiscal_year,form_type,table_index,section_title,section_path,chunk_uid,snippet
0,1,AAPL_10-K_2024::text::55,False,text_chunk,AAPL,2024,10-K,None,Basis of Presentation and Preparation,Item 6. [Reserved] > Basis of Presentation and...,None,Item 6. [Reserved] > Basis of Presentation and...
1,2,AAPL_10-K_2024::text::41,True,text_chunk,AAPL,2024,10-K,None,Fiscal Period,Item 6. [Reserved] > Fiscal Period,None,Item 6. [Reserved] > Fiscal Period\n\nThe Comp...
2,3,AAPL_10-K_2024::text::0,False,text_chunk,AAPL,2024,10-K,None,Company Background,Item 1. Business > Company Background,None,Item 1. Business > Company Background\n\nThe C...
